# Normalizing flows

_Modern Deep Learning & AI_

**Bend a plain Gaussian through an invertible map into any shape, and read off the exact probability.**

A normalizing flow builds a complex probability distribution out of a simple one.

     Start with an easy density, a standard Gaussian. Pass each sample through an invertible transform $g$. The Gaussian gets stretched and folded into a rich, multi-peaked shape.

---

This notebook is a practice scaffold for the **Normalizing flows** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

class CouplingLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.half = dim // 2
        # net reads the untouched half, outputs scale (log s) and shift (t)
        self.net = nn.Sequential(nn.Linear(self.half, 16), nn.ReLU(),
                                 nn.Linear(16, 2 * (dim - self.half)))
    def forward(self, x):
        xa, xb = x[:, :self.half], x[:, self.half:]   # split
        log_s, t = self.net(xa).chunk(2, dim=1)
        log_s = torch.tanh(log_s)                      # stabilize the scale
        yb = xb * torch.exp(log_s) + t                 # affine transform on xb
        y = torch.cat([xa, yb], dim=1)
        log_det = log_s.sum(dim=1)                      # Jacobian log-det
        return y, log_det

torch.manual_seed(0)
dim = 4
layer = CouplingLayer(dim)
x = torch.randn(8, dim)                                # base samples ~ N(0, I)
y, log_det = layer(x)
# exact log-likelihood under the flow = base log-prob + log|det Jacobian|
base_logp = (-0.5 * y.pow(2) - 0.5 * torch.log(torch.tensor(2 * 3.14159))).sum(1)
log_prob = base_logp + log_det
print("y shape:", y.shape, "log_det[0]:", round(log_det[0].item(), 4))
print("log_prob[0]:", round(log_prob[0].item(), 4))

## Visualize the data & results

_Can an invertible map turn one Gaussian hump into the real bimodal spread of digits 0 and 1?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# REAL bimodal target: PCA component-1 scores of digits 0 and 1
digits = load_digits()
Z = PCA(n_components=2).fit_transform(digits.data / 16.0)
mask = (digits.target == 0) | (digits.target == 1)
v = Z[mask, 0]
v = (v - v.mean()) / v.std()
hist, edges = np.histogram(v, bins=18, density=True)
centers = 0.5 * (edges[:-1] + edges[1:])

# flow: push a base Gaussian through x = u + sep*tanh(u) (change of variables)
sep = 1.5
u = np.linspace(-3.5, 3.5, 41)
pu = np.exp(-0.5 * u ** 2) / np.sqrt(2 * np.pi)
x = u + sep * np.tanh(u)                          # invertible transform g(u)
g_prime = 1.0 + sep * (1.0 - np.tanh(u) ** 2)
px = pu / np.abs(g_prime)                         # transformed density

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(u, pu, color="#4ea1ff", label="base p_u(u): Gaussian")
ax.plot(x, px, color="#7ee787", label="flow p_x(x): two peaks")
ax.plot(centers, hist, color="#ffb454", label="real digit 0 and 1 scores")
ax.set_xlabel("value"); ax.set_ylabel("probability density")
ax.set_title("flow matches the real bimodal target"); ax.legend()
plt.show()